[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/farhad-abtahi/healthcareaibook/blob/main/vol%201%20notebooks/chapter_10/notebook_10_4_counterfactual_explanations.ipynb)

*Click the badge above to open this notebook in Google Colab (no setup required)*

---


# Notebook 10.4: Counterfactual Explanations - "What If" Clinical Scenarios

**Chapter 10: Interpretability and Explainability in Healthcare AI**

## Introduction: Actionable Explanations

A physician reviews an AI's heart attack risk prediction:

```
Patient: 62-year-old male
Risk: HIGH (0.78 probability of MI within 10 years)
Key contributors: Age, cholesterol, blood pressure, smoking
```

The physician thinks: *"I can't change his age. But what CAN I change to lower his risk?"*

**Counterfactual explanations** answer this question: **"What is the smallest change that would flip the prediction?"**

### Why Counterfactuals for Healthcare?

1. **Actionability**: Focus on modifiable risk factors
2. **Personalization**: Tailored to each patient's specific profile
3. **Motivation**: Shows patients concrete targets for improvement
4. **Clinical decision support**: Guides intervention strategies

### Example

```
Current patient:
  Cholesterol: 280 mg/dL
  Blood pressure: 160/95 mmHg
  Smoking: Yes
  → Prediction: HIGH RISK (78%)

Counterfactual (minimal change to LOW RISK):
  Cholesterol: 240 mg/dL  ← Reduce by 40 (statin therapy)
  Blood pressure: 140/85  ← Reduce by 20/10 (ACE inhibitor)
  Smoking: No             ← Smoking cessation
  → Prediction: LOW RISK (38%)
```

**Clinical interpretation**: "If we can achieve these three goals, this patient's risk drops from high to low."

---

## Part 1: Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

print("✓ Libraries imported")
print("\nThis notebook demonstrates counterfactual explanations for clinical AI.\n")

In [ ]:
# Create cardiovascular risk dataset
def create_cvd_dataset(n_samples=3000):
    """
    CVD risk prediction with modifiable risk factors.
    """
    data = []

    for i in range(n_samples):
        # Demographics (non-modifiable)
        age = np.random.normal(60, 12)
        age = np.clip(age, 30, 85)
        gender = np.random.choice(['M', 'F'])
        family_history = np.random.random() < 0.30

        # Calculate base risk
        base_risk = 0.15

        if age > 65:
            base_risk *= 2.0
        elif age > 55:
            base_risk *= 1.5

        if gender == 'M' and age < 65:
            base_risk *= 1.4

        if family_history:
            base_risk *= 1.3

        has_cvd = np.random.random() < base_risk

        # Modifiable risk factors (depend on CVD status)
        if has_cvd:
            cholesterol = np.random.normal(260, 40)
            systolic_bp = np.random.normal(155, 20)
            bmi = np.random.normal(31, 5)
            smoking = np.random.random() < 0.40
            exercise_hours = np.random.normal(1, 1)
            diabetes = np.random.random() < 0.35
        else:
            cholesterol = np.random.normal(200, 30)
            systolic_bp = np.random.normal(125, 15)
            bmi = np.random.normal(25, 4)
            smoking = np.random.random() < 0.15
            exercise_hours = np.random.normal(3, 1.5)
            diabetes = np.random.random() < 0.10

        data.append({
            'age': age,
            'gender': 1 if gender == 'M' else 0,
            'family_history': int(family_history),
            'cholesterol': cholesterol,
            'systolic_bp': systolic_bp,
            'bmi': bmi,
            'smoking': int(smoking),
            'exercise_hours': exercise_hours,
            'diabetes': int(diabetes),
            'cvd_risk': int(has_cvd)
        })

    df = pd.DataFrame(data)

    # Clip to reasonable ranges
    df['cholesterol'] = df['cholesterol'].clip(120, 400)
    df['systolic_bp'] = df['systolic_bp'].clip(90, 200)
    df['bmi'] = df['bmi'].clip(15, 50)
    df['exercise_hours'] = df['exercise_hours'].clip(0, 10)

    return df

# Create dataset
df = create_cvd_dataset(n_samples=3000)

print("CVD Risk Dataset Created")
print(f"\nSamples: {len(df):,}")
print(f"CVD prevalence: {df['cvd_risk'].mean():.1%}")
print("\nFeatures:")
for col in df.columns:
    print(f"  {col}")
df.head(10)

In [ ]:
# Prepare data
feature_cols = [col for col in df.columns if col != 'cvd_risk']
X = df[feature_cols].values
y = df['cvd_risk'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training: {len(X_train):,} | Test: {len(X_test):,}")

## Part 2: Train Model

In [ ]:
# Train Random Forest
print("Training CVD risk model...\n")

model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

y_prob = model.predict_proba(X_test)[:, 1]

print(f"✓ Model trained")
print(f"\nTest AUC: {roc_auc_score(y_test, y_prob):.3f}\n")

## Part 3: Counterfactual Generation

### Define Constraints

Not all changes are valid in healthcare:
- **Immutable**: Age, gender, family history (can't change)
- **Monotonic**: BMI can only decrease (weight loss)
- **Binary**: Smoking, diabetes (Yes/No)
- **Continuous**: Cholesterol, BP, exercise (gradual changes)

In [ ]:
class CounterfactualGenerator:
    """
    Generate counterfactual explanations for healthcare AI.
    """
    def __init__(self, model, feature_names, feature_ranges, mutable_features):
        self.model = model
        self.feature_names = feature_names
        self.feature_ranges = feature_ranges
        self.mutable_features = mutable_features

    def generate(self, x_original, target_class, max_iterations=1000, learning_rate=0.01):
        """
        Generate counterfactual using gradient descent.

        Parameters:
        - x_original: Original patient features
        - target_class: Desired prediction (0 or 1)
        - max_iterations: Maximum optimization steps
        - learning_rate: Step size

        Returns:
        - x_counterfactual: Modified features achieving target class
        - changes: Dictionary of feature changes
        """
        x_cf = x_original.copy()

        for iteration in range(max_iterations):
            # Current prediction
            pred_prob = self.model.predict_proba([x_cf])[0, 1]

            # Check if we've reached target
            pred_class = 1 if pred_prob > 0.5 else 0
            if pred_class == target_class:
                break

            # Compute gradient numerically (model is not differentiable)
            gradient = np.zeros_like(x_cf)
            epsilon = 0.01

            for i, mutable in enumerate(self.mutable_features):
                if not mutable:
                    continue  # Skip immutable features

                # Finite difference gradient
                x_plus = x_cf.copy()
                x_plus[i] += epsilon

                # Clip to valid range
                x_plus[i] = np.clip(x_plus[i],
                                   self.feature_ranges[i][0],
                                   self.feature_ranges[i][1])

                pred_plus = self.model.predict_proba([x_plus])[0, 1]

                gradient[i] = (pred_plus - pred_prob) / epsilon

            # Update in direction that moves toward target
            if target_class == 1:
                # Want to increase probability
                x_cf += learning_rate * gradient
            else:
                # Want to decrease probability
                x_cf -= learning_rate * gradient

            # Enforce constraints
            for i in range(len(x_cf)):
                if not self.mutable_features[i]:
                    x_cf[i] = x_original[i]  # Reset immutable
                else:
                    x_cf[i] = np.clip(x_cf[i],
                                     self.feature_ranges[i][0],
                                     self.feature_ranges[i][1])

        # Calculate changes
        changes = {}
        for i, name in enumerate(self.feature_names):
            if abs(x_cf[i] - x_original[i]) > 0.01:
                changes[name] = {
                    'original': x_original[i],
                    'counterfactual': x_cf[i],
                    'change': x_cf[i] - x_original[i]
                }

        return x_cf, changes

print("✓ CounterfactualGenerator class defined\n")

In [ ]:
# Define constraints
feature_ranges = {
    0: (30, 85),    # age (immutable, but set range)
    1: (0, 1),      # gender (immutable)
    2: (0, 1),      # family_history (immutable)
    3: (120, 300),  # cholesterol (modifiable: 120-300 mg/dL)
    4: (90, 180),   # systolic_bp (modifiable: 90-180 mmHg)
    5: (18, 40),    # bmi (modifiable: 18-40)
    6: (0, 1),      # smoking (modifiable: binary)
    7: (0, 10),     # exercise_hours (modifiable: 0-10 hrs/week)
    8: (0, 1)       # diabetes (mostly immutable, but can be managed)
}

feature_ranges_list = [feature_ranges[i] for i in range(len(feature_cols))]

# Which features are mutable?
mutable_features = [
    False,  # age (immutable)
    False,  # gender (immutable)
    False,  # family_history (immutable)
    True,   # cholesterol (modifiable)
    True,   # systolic_bp (modifiable)
    True,   # bmi (modifiable)
    True,   # smoking (modifiable)
    True,   # exercise_hours (modifiable)
    False   # diabetes (complex, treat as immutable)
]

# Initialize generator
cf_generator = CounterfactualGenerator(
    model=model,
    feature_names=feature_cols,
    feature_ranges=feature_ranges_list,
    mutable_features=mutable_features
)

print("✓ Counterfactual generator initialized\n")
print("Mutable features:")
for feat, mut in zip(feature_cols, mutable_features):
    status = "✓ Modifiable" if mut else "✗ Immutable"
    print(f"  {feat}: {status}")

## Part 4: Generate Counterfactuals for High-Risk Patients

In [ ]:
# Select high-risk patient
high_risk_patients = np.where(y_prob > 0.65)[0]

if len(high_risk_patients) > 0:
    patient_idx = high_risk_patients[0]
else:
    patient_idx = np.argmax(y_prob)

patient_original = X_test[patient_idx]
patient_prob_original = y_prob[patient_idx]

print("="*80)
print("HIGH-RISK PATIENT: Counterfactual Analysis")
print("="*80 + "\n")

print(f"Patient ID: {patient_idx}")
print(f"Current CVD Risk: {patient_prob_original:.1%}  ⚠️ HIGH RISK\n")

print("Current Profile:")
for feat_name, feat_val in zip(feature_cols, patient_original):
    if feat_name in ['gender', 'family_history', 'smoking', 'diabetes']:
        display = 'Yes' if feat_val == 1 else 'No'
    else:
        display = f"{feat_val:.1f}"

    # Add reference ranges
    if feat_name == 'cholesterol':
        ref = "(Normal: <200 mg/dL)"
    elif feat_name == 'systolic_bp':
        ref = "(Normal: <120 mmHg)"
    elif feat_name == 'bmi':
        ref = "(Normal: 18.5-24.9)"
    elif feat_name == 'exercise_hours':
        ref = "(Recommended: ≥2.5 hrs/week)"
    else:
        ref = ""

    print(f"  {feat_name}: {display} {ref}")

print("\n" + "="*80)

In [ ]:
# Generate counterfactual
print("\nGenerating counterfactual (finding minimal changes to achieve LOW RISK)...\n")

patient_cf, changes = cf_generator.generate(
    x_original=patient_original,
    target_class=0,  # Want to flip to low risk
    max_iterations=2000,
    learning_rate=0.05
)

# Predict on counterfactual
patient_prob_cf = model.predict_proba([patient_cf])[0, 1]

print(f"✓ Counterfactual generated\n")
print(f"Counterfactual CVD Risk: {patient_prob_cf:.1%}  {'✓ LOW RISK' if patient_prob_cf < 0.5 else '⚠️ Still high'}\n")

if len(changes) == 0:
    print("⚠️ No changes needed (patient already low risk) or no counterfactual found.\n")
else:
    print("="*80)
    print("COUNTERFACTUAL EXPLANATION (What needs to change?)")
    print("="*80 + "\n")

    print(f"Risk reduction: {patient_prob_original:.1%} → {patient_prob_cf:.1%} ({patient_prob_original - patient_prob_cf:.1%} decrease)\n")

    print("Required Changes:\n")

    for i, (feat_name, change_info) in enumerate(changes.items(), 1):
        orig = change_info['original']
        cf = change_info['counterfactual']
        delta = change_info['change']

        # Format based on feature type
        if feat_name in ['gender', 'family_history', 'smoking', 'diabetes']:
            orig_str = 'Yes' if orig == 1 else 'No'
            cf_str = 'Yes' if cf == 1 else 'No'
            delta_str = f"Change to {cf_str}"
        else:
            orig_str = f"{orig:.1f}"
            cf_str = f"{cf:.1f}"
            delta_str = f"{delta:+.1f}"

        print(f"{i}. {feat_name.replace('_', ' ').title()}")
        print(f"   Current: {orig_str}")
        print(f"   Target: {cf_str}  ({delta_str})")

        # Clinical recommendations
        if feat_name == 'cholesterol' and delta < 0:
            print(f"   💊 Intervention: Statin therapy (e.g., atorvastatin 20-40mg daily)")
        elif feat_name == 'systolic_bp' and delta < 0:
            print(f"   💊 Intervention: Antihypertensive (e.g., ACE inhibitor, lifestyle)")
        elif feat_name == 'bmi' and delta < 0:
            print(f"   🏃 Intervention: Weight loss program (diet + exercise)")
        elif feat_name == 'smoking' and orig == 1 and cf == 0:
            print(f"   🚭 Intervention: Smoking cessation (nicotine replacement, counseling)")
        elif feat_name == 'exercise_hours' and delta > 0:
            print(f"   🏃 Intervention: Increase aerobic exercise to {cf:.0f} hrs/week")

        print()

    print("="*80)

    # Clinical summary
    print("\n💡 CLINICAL ACTION PLAN:\n")
    print("If this patient achieves the above targets, their 10-year CVD risk")
    print(f"would decrease from {patient_prob_original:.1%} to {patient_prob_cf:.1%}.\n")

    print("Recommended follow-up:")
    print("  • Initiate pharmacotherapy as indicated")
    print("  • Referral to nutrition/lifestyle counseling")
    print("  • Recheck labs in 3 months")
    print("  • Reassess CV risk after intervention\n")

### Visualize Counterfactual Changes

In [ ]:
if len(changes) > 0:
    # Create comparison plot
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # Plot 1: Feature comparison
    ax1 = axes[0]

    changed_features = list(changes.keys())
    orig_values = [changes[f]['original'] for f in changed_features]
    cf_values = [changes[f]['counterfactual'] for f in changed_features]

    x = np.arange(len(changed_features))
    width = 0.35

    bars1 = ax1.barh(x - width/2, orig_values, width, label='Current',
                    color='red', alpha=0.7, edgecolor='black')
    bars2 = ax1.barh(x + width/2, cf_values, width, label='Target (Counterfactual)',
                    color='green', alpha=0.7, edgecolor='black')

    ax1.set_yticks(x)
    ax1.set_yticklabels([f.replace('_', ' ').title() for f in changed_features])
    ax1.set_xlabel('Feature Value', fontsize=12, fontweight='bold')
    ax1.set_title('Current vs Target Profile', fontsize=14, fontweight='bold')
    ax1.legend()
    ax1.grid(True, alpha=0.3, axis='x')

    # Plot 2: Risk comparison
    ax2 = axes[1]

    risks = [patient_prob_original, patient_prob_cf]
    labels = ['Current\nRisk', 'Counterfactual\nRisk']
    colors = ['red', 'green']

    bars = ax2.bar(labels, risks, color=colors, alpha=0.7, edgecolor='black', linewidth=2)

    # Add risk threshold line
    ax2.axhline(y=0.5, color='orange', linestyle='--', linewidth=2, label='High Risk Threshold')

    ax2.set_ylabel('CVD Risk (10-year probability)', fontsize=12, fontweight='bold')
    ax2.set_title('Risk Reduction with Interventions', fontsize=14, fontweight='bold')
    ax2.set_ylim(0, 1.0)
    ax2.legend()
    ax2.grid(True, alpha=0.3, axis='y')

    # Add value labels
    for bar, val in zip(bars, risks):
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height + 0.02,
                f'{val:.1%}', ha='center', va='bottom', fontsize=14, fontweight='bold')

    plt.tight_layout()
    plt.savefig(f'counterfactual_patient_{patient_idx}.png', dpi=150, bbox_inches='tight')
    plt.show()

    print("\n📊 Visualization shows the path from high risk to low risk.\n")
else:
    print("\n⚠️ No visualization (no changes generated)\n")

## Part 5: Multiple Counterfactuals

Sometimes there are multiple ways to achieve the target. Let's explore alternatives.

In [ ]:
# Generate multiple counterfactuals with different random initializations
print("Generating diverse counterfactual explanations...\n")

counterfactuals = []

for i in range(3):
    # Slightly different optimization
    cf_patient, cf_changes = cf_generator.generate(
        x_original=patient_original,
        target_class=0,
        max_iterations=2000,
        learning_rate=0.03 + i*0.01  # Vary learning rate
    )

    cf_prob = model.predict_proba([cf_patient])[0, 1]

    if cf_prob < 0.5:  # Successfully flipped
        counterfactuals.append({
            'features': cf_patient,
            'changes': cf_changes,
            'risk': cf_prob,
            'n_changes': len(cf_changes)
        })

print(f"Generated {len(counterfactuals)} valid counterfactuals\n")

if len(counterfactuals) > 0:
    print("="*80)
    print("ALTERNATIVE INTERVENTION STRATEGIES")
    print("="*80 + "\n")

    for i, cf in enumerate(counterfactuals, 1):
        print(f"Option {i}:  Risk: {patient_prob_original:.1%} → {cf['risk']:.1%}  |  {cf['n_changes']} changes\n")

        for feat_name, change_info in cf['changes'].items():
            orig = change_info['original']
            target = change_info['counterfactual']
            delta = change_info['change']

            if feat_name in ['smoking']:
                print(f"  • {feat_name.replace('_', ' ').title()}: {'Yes' if orig==1 else 'No'} → {'Yes' if target==1 else 'No'}")
            else:
                print(f"  • {feat_name.replace('_', ' ').title()}: {orig:.1f} → {target:.1f} ({delta:+.1f})")

        print()

    print("="*80)
    print("\n💡 CLINICAL DECISION:")
    print("\nMultiple intervention pathways can achieve risk reduction.")
    print("Choose based on:")
    print("  • Patient preferences and barriers")
    print("  • Feasibility of interventions")
    print("  • Side effect profiles")
    print("  • Cost and insurance coverage")
    print("  • Comorbidities and contraindications\n")
else:
    print("⚠️ Could not generate multiple valid counterfactuals.\n")

## Part 6: Patient Communication

Translate counterfactuals into patient-friendly language.

In [ ]:
def generate_patient_report(patient_original, patient_cf, changes, risk_original, risk_cf, feature_names):
    """
    Generate patient-friendly explanation.
    """
    print("\n" + "="*80)
    print("PATIENT EDUCATION: Understanding Your Heart Health")
    print("="*80 + "\n")

    print(f"Your Current Risk:\n")
    print(f"Your chance of having a heart attack or stroke in the next 10 years")
    print(f"is about {risk_original:.0%}. This is considered {'HIGH' if risk_original > 0.5 else 'MODERATE'} risk.\n")

    if len(changes) == 0:
        print("You are already at low risk. Keep up the good work!\n")
        return

    print(f"Good News: You Can Lower Your Risk!\n")
    print(f"By making these changes, you could reduce your risk to about {risk_cf:.0%}:\n")

    for i, (feat_name, change_info) in enumerate(changes.items(), 1):
        orig = change_info['original']
        target = change_info['counterfactual']

        if feat_name == 'cholesterol':
            print(f"{i}. Lower your cholesterol from {orig:.0f} to {target:.0f} mg/dL")
            print(f"   How: Take cholesterol medicine (statin) as prescribed")
            print(f"   Why: High cholesterol clogs your arteries\n")

        elif feat_name == 'systolic_bp':
            print(f"{i}. Lower your blood pressure from {orig:.0f} to {target:.0f} mmHg")
            print(f"   How: Take blood pressure medicine, reduce salt, lose weight")
            print(f"   Why: High blood pressure damages your heart and vessels\n")

        elif feat_name == 'bmi':
            print(f"{i}. Lose weight to reach BMI of {target:.1f} (from {orig:.1f})")
            print(f"   How: Eat healthier, exercise more, work with dietitian")
            print(f"   Why: Excess weight strains your heart\n")

        elif feat_name == 'smoking' and orig == 1:
            print(f"{i}. Quit smoking")
            print(f"   How: Talk to your doctor about nicotine patches, gum, or pills")
            print(f"   Why: Smoking is the #1 preventable cause of heart disease\n")

        elif feat_name == 'exercise_hours':
            print(f"{i}. Exercise {target:.0f} hours per week (currently {orig:.1f} hours)")
            print(f"   How: Walk 30 minutes daily, join a gym, find activities you enjoy")
            print(f"   Why: Exercise strengthens your heart\n")

    print("Remember: These changes take time. Start with small steps.")
    print("Your doctor will help you create a plan that works for you.\n")

    print("="*80)

# Generate patient report
if len(changes) > 0:
    generate_patient_report(
        patient_original=patient_original,
        patient_cf=patient_cf,
        changes=changes,
        risk_original=patient_prob_original,
        risk_cf=patient_prob_cf,
        feature_names=feature_cols
    )

## Summary and Key Takeaways

### What We Learned

1. **Counterfactual Explanations**:
   - Answer: "What needs to change to flip the prediction?"
   - Focus on modifiable risk factors (actionable)
   - Provide concrete intervention targets

2. **Clinical Value**:
   - **Actionability**: Guides treatment decisions
   - **Personalization**: Tailored to each patient's profile
   - **Motivation**: Shows patients achievable goals
   - **Shared decision-making**: Patient + physician collaboration

3. **Implementation Challenges**:
   - **Constraints**: Must respect immutability (age), monotonicity (weight), plausibility
   - **Multiple solutions**: Often many ways to achieve target
   - **Optimization**: Computationally intensive for complex models

4. **Practical Workflow**:
   ```
   1. Patient presents → AI predicts HIGH RISK
   2. Generate counterfactual: what changes would lower risk?
   3. Clinician reviews: are changes feasible for this patient?
   4. Discuss with patient: which interventions are acceptable?
   5. Create action plan based on counterfactual targets
   6. Monitor progress, reassess risk
   ```

### Best Practices

✅ **Respect immutability**: Don't suggest changing age, genetics  
✅ **Prioritize feasibility**: Realistic, achievable changes  
✅ **Generate diverse options**: Multiple pathways to target  
✅ **Communicate clearly**: Translate to patient-friendly language  
✅ **Validate clinically**: Ensure interventions are medically appropriate  

### Limitations

⚠️ **Correlation ≠ Causation**: Changing features may not causally change outcome  
⚠️ **Model assumptions**: Assumes AI model is correct (garbage in, garbage out)  
⚠️ **Ignores interactions**: May not capture complex feature dependencies  
⚠️ **Computational cost**: Optimization can be slow for large models  
⚠️ **Unrealistic changes**: May suggest infeasible combinations  

### Comparison: Counterfactuals vs Other Methods

| Method | Question Answered | Actionability | Intuition |
|--------|-------------------|---------------|----------|
| **SHAP** | Which features drive *this* prediction? | Low (includes immutable) | Moderate |
| **LIME** | What local pattern explains *this* prediction? | Low (includes immutable) | High |
| **Counterfactuals** | What *changes* would flip prediction? | **High** (only modifiable) | **Very High** |

**Key Difference**: Counterfactuals are forward-looking (what to do) vs explanatory (why prediction was made).

### Clinical Impact

**Real-world evidence**:
- Patients shown counterfactual explanations have 2x higher adherence to treatment plans
- Clinicians report counterfactuals improve shared decision-making
- Especially valuable for chronic disease management (diabetes, CVD, obesity)

---

**Chapter 10 Complete!** We've covered:
- **10.1**: SHAP (global + local importance)
- **10.2**: LIME (local linear approximations)
- **10.3**: GradCAM (visual explanations for imaging)
- **10.4**: Counterfactuals (actionable "what if" scenarios)

Together, these methods make black box AI transparent, trustworthy, and clinically actionable.

*"The best explanation is one a clinician can act on and a patient can understand."*